# Production-Pattern LoRA Fine-Tuning of Stable Diffusion 1.5

**Goal:** learn the *real* fine-tuning pipeline that production teams use for diffusion models — not the 5-photo DreamBooth toy, but caption-based LoRA fine-tuning on a dataset, the way you'd fine-tune a model on a client's product catalog, a brand's visual style, or a domain-specific image set.

**Why SD 1.5 and not SDXL/FLUX?** On a free Colab T4 (15GB VRAM), SDXL is tight and FLUX (needs 24–40GB even with QLoRA tricks) won't fit at all. SD 1.5 fits comfortably with room to experiment, and **the entire pipeline — dataset → tokenizer/text-encoder → VAE → UNet → noise scheduler → LoRA injection → training loop → inference** is architecturally identical to SDXL and FLUX. Swap the model name and a few dimensions later and you're fine-tuning the bigger models with the same mental model.

**Dataset:** `reach-vb/pokemon-blip-captions` (mirror of the original Lambda Labs dataset, which was DMCA'd). 833 images, each with a real auto-generated text caption (via BLIP) — not a single trigger word. This is the **caption-based fine-tuning pattern**, which is what you use when you have a real dataset of (image, description) pairs, as opposed to DreamBooth's "5 photos of one subject + one trigger word" pattern.

**What you'll actually learn:**
1. How a diffusion model's training objective works (predicting noise, not pixels)
2. Why we only fine-tune the UNet's attention layers via LoRA, not the whole model
3. The exact production stack: `diffusers` + `peft` + `accelerate`
4. How to monitor a training run for collapse/divergence (the failure mode that wastes the most GPU-hours in real jobs)
5. How to load your adapter back and generate with it — and how this generalizes to swapping in your own image dataset

---
**Runtime → Change runtime type → T4 GPU** before running anything below.

## 0. Setup: install the exact production stack

These are the same libraries used in real fine-tuning jobs at companies doing this in production:
- `diffusers` — HuggingFace's diffusion model library, has the training script we'll adapt
- `peft` — Parameter-Efficient Fine-Tuning (this is what implements LoRA)
- `accelerate` — handles mixed precision, device placement, gradient accumulation
- `bitsandbytes` — 8-bit optimizer, cuts optimizer memory significantly
- `transformers` — CLIP text encoder lives here

**Why version conflicts happen on Colab specifically, and how this notebook avoids them:**
1. Colab's base image ships older versions of some of these libraries pre-imported in its base environment. If you `pip install` a different version without restarting, Python can end up with a mix of old and new code in memory — this is the most common cause of confusing errors that don't match the installed version.
2. `diffusers` releases now hard-require `peft>=0.17`. Any tutorial pinning an older `peft` against a newer `diffusers` (or vice versa) will fail on import with a clear `ImportError` naming the version requirement — if you ever see that error, it's telling you exactly which package to bump.
3. We intentionally do not reinstall `torch`. Colab's preinstalled torch is already matched to the GPU driver in your runtime. Installing your own torch/CUDA combination on top is the single most common way to break a Colab GPU environment.

**After running the install cell below, go to `Runtime -> Restart session` once, then continue from the cell after that (do not re-run the install cell).** This clears any old versions Colab had pre-loaded before your pip install ran.

In [ ]:
# IMPORTANT: we deliberately do NOT touch torch/torchvision here.
# Colab's preinstalled torch is already CUDA-matched to the GPU driver in this runtime
# (Colab updates this for you) -- reinstalling torch yourself is the #1 cause of
# "works in tutorial, breaks in my Colab" version-conflict hell. Let it be.
#
# We pin the HF stack to a tested-compatible combo instead of leaving it fully open,
# because `diffusers` and `peft` move fast and recent `diffusers` releases hard-require
# `peft>=0.17`. Pinning here = reproducible; leaving unpinned = whatever shipped today.
!pip install -q -U \
    "diffusers==0.38.0" \
    "transformers==4.57.1" \
    "accelerate==1.13.0" \
    "peft==0.17.1" \
    "bitsandbytes==0.49.2" \
    "datasets==3.6.0" \
    ftfy tensorboard

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else "n/a")

# Sanity check the install actually worked before going any further --
# if this cell errors, RESTART THE RUNTIME (Runtime -> Restart session) and run again.
# Colab sometimes has older versions of these libs already imported in memory from
# its base image, and pip installing over them without a restart causes ghost conflicts.
import diffusers, transformers, peft, accelerate
print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

## 1. Load and inspect the dataset

**Production habit #1: always look at your data before training on it.** A huge fraction of real fine-tuning failures trace back to a dataset that has duplicate images, broken captions, or a resolution mismatch nobody caught. We're going to actually look at samples, check caption length distribution, and check image sizes before writing a single line of training code.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("reach-vb/pokemon-blip-captions", split="train")
print(dataset)
print("\nExample row keys:", dataset[0].keys())
print("Example caption:", dataset[0]["text"])
print("Example image size:", dataset[0]["image"].size)

In [ ]:
import matplotlib.pyplot as plt

# Visual sanity check — always do this before training
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, ax in enumerate(axes):
    idx = i * 150
    ax.imshow(dataset[idx]["image"])
    ax.set_title(dataset[idx]["text"][:40] + "...", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

# Caption length distribution — if this is wildly skewed, your tokenizer truncation will hurt you
caption_lengths = [len(t["text"].split()) for t in dataset]
print("Caption word count — min/mean/max:", min(caption_lengths), sum(caption_lengths)/len(caption_lengths), max(caption_lengths))

# Image size consistency — if these vary wildly, resizing strategy matters more
sizes = set(t["image"].size for t in dataset)
print("Unique image sizes (sample):", list(sizes)[:5], "..." if len(sizes) > 5 else "")

## 2. Load the base model components individually

**This is the part most tutorials skip and most people don't actually understand.** A diffusion pipeline isn't one model — it's four separate pieces wired together. Understanding this is the difference between someone who can run a script and someone who can debug a training run when it goes wrong.

| Component | Job | Do we train it? |
|---|---|---|
| **Tokenizer** | Text → token IDs | No (not a model) |
| **Text Encoder** (CLIP) | Token IDs → embeddings the UNet can use as conditioning | No — frozen |
| **VAE** | Compresses images to a smaller latent space and back | No — frozen |
| **UNet** | The actual denoiser — predicts the noise in a noisy latent, conditioned on the text embedding | **Yes — via LoRA adapters injected into its attention layers** |
| **Scheduler** | The noise schedule math (how much noise at each timestep, how to step backward) | No (not a model, just math) |

This is *why* LoRA fine-tuning of diffusion models is cheap: the text encoder and VAE (which together are a large chunk of the parameters) stay completely frozen. We only touch the UNet, and even then only small low-rank matrices inside its attention blocks.

In [ ]:
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from transformers import CLIPTextModel, CLIPTokenizer

# Note: the original `runwayml/stable-diffusion-v1-5` repo was deleted from the Hub in 2024
# when RunwayML stopped maintaining their HF org. This is the official, identical mirror.
MODEL_NAME = "stable-diffusion-v1-5/stable-diffusion-v1-5"

tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL_NAME, subfolder="text_encoder")
vae = AutoencoderKL.from_pretrained(MODEL_NAME, subfolder="vae")
unet = UNet2DConditionModel.from_pretrained(MODEL_NAME, subfolder="unet")
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_NAME, subfolder="scheduler")

# Freeze everything by default — we'll selectively unfreeze LoRA params on the UNet only
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

print("UNet params (millions):", sum(p.numel() for p in unet.parameters()) / 1e6)
print("Text encoder params (millions):", sum(p.numel() for p in text_encoder.parameters()) / 1e6)
print("VAE params (millions):", sum(p.numel() for p in vae.parameters()) / 1e6)

## 3. Inject LoRA into the UNet's attention layers

This is the core trick. Instead of updating the full weight matrix $W$ of each attention projection (query/key/value/output), we freeze $W$ and add a small update:

$$ W' = W + BA $$

where $B$ and $A$ are small low-rank matrices (rank $r$, typically 4–32). If $W$ is $d \times d$, then $A$ is $r \times d$ and $B$ is $d \times r$ — so instead of $d^2$ trainable parameters you get $2rd$, which for $r \ll d$ is a tiny fraction.

We target the attention projection layers (`to_q`, `to_k`, `to_v`, `to_out.0`) because that's where the model decides *what to attend to* given the text conditioning — exactly the part you need to change to teach the model a new visual concept/style.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,                       # rank — higher = more capacity, more memory. 16 is a solid default per current best practice
    lora_alpha=16,               # scaling factor, commonly set equal to r
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],  # attention projections in the UNet
    lora_dropout=0.0,
    bias="none",
)

unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

You should see something like **~0.7-1% of UNet parameters are trainable.** That's the entire point of LoRA — a 0.86B-parameter UNet, but you're only training a few million parameters. This is also *why* the resulting adapter file is tiny (a few MB instead of several GB) and why you can swap multiple LoRA adapters in and out of the same base model at inference time.

## 4. Build the training dataset pipeline

Production pattern: pre-process once, cache tensors, don't redo CLIP tokenization or image transforms on every step. We:
1. Resize/center-crop images to 512×512 (SD 1.5's native resolution)
2. Normalize to [-1, 1] (what the VAE expects)
3. Tokenize captions with truncation/padding to a fixed length

In [ ]:
from torchvision import transforms
import torch

resolution = 512

image_transforms = transforms.Compose([
    transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(resolution),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),  # maps [0,1] -> [-1,1]
])

def preprocess(examples):
    images = [image.convert("RGB") for image in examples["image"]]
    examples["pixel_values"] = [image_transforms(image) for image in images]
    examples["input_ids"] = tokenizer(
        examples["text"],
        max_length=tokenizer.model_max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).input_ids
    return examples

train_dataset = dataset.with_transform(preprocess)

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    input_ids = torch.stack([example["input_ids"] for example in examples])
    return {"pixel_values": pixel_values, "input_ids": input_ids}

from torch.utils.data import DataLoader

BATCH_SIZE = 1          # T4 constraint — we use gradient accumulation to compensate
GRAD_ACCUM_STEPS = 4    # effective batch size = 4

train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2
)

print("Number of batches per epoch:", len(train_dataloader))

## 5. The training loop — understanding the actual objective

This is the part that separates "I ran a script" from "I understand fine-tuning." Here's exactly what happens on every training step:

1. **Encode the image to latent space** via the frozen VAE encoder (this is why we work in latent space, not pixel space — it's ~8x smaller, hence "latent diffusion")
2. **Sample random noise** and a **random timestep** $t$
3. **Add that noise to the latent** according to the schedule at timestep $t$ — this gives us a "noisy latent"
4. **Encode the caption** via the frozen CLIP text encoder → conditioning embedding
5. **Feed the noisy latent + timestep + text embedding into the UNet (with LoRA active)** — the UNet's job is to predict *what noise was added*
6. **Loss = MSE between predicted noise and actual noise**
7. Backprop — gradients only flow into the LoRA matrices, since everything else is frozen

The model never sees clean target images directly during loss computation — it learns by getting good at **undoing noise conditioned on a caption**. This is why diffusion models are called "denoising" models, and it's the single most important conceptual thing to internalize before touching the hyperparameters.

In [ ]:
import torch.nn.functional as F
from accelerate import Accelerator
from accelerate.utils import set_seed
from torch.optim import AdamW
from diffusers.optimization import get_scheduler
import bitsandbytes as bnb

set_seed(42)

accelerator = Accelerator(mixed_precision="fp16", gradient_accumulation_steps=GRAD_ACCUM_STEPS)
device = accelerator.device

weight_dtype = torch.float16
vae.to(device, dtype=weight_dtype)
text_encoder.to(device, dtype=weight_dtype)
unet.to(device)  # LoRA params stay fp32 for training stability; base UNet weights are frozen fp16

# 8-bit AdamW: cuts optimizer state memory ~4x vs standard AdamW, standard production trick for tight VRAM
trainable_params = [p for p in unet.parameters() if p.requires_grad]
optimizer = bnb.optim.AdamW8bit(trainable_params, lr=1e-4)

NUM_EPOCHS = 15  # ~833 images, small dataset, LoRA converges fast — watch the loss curve, don't just trust this number
max_train_steps = (len(train_dataloader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS

lr_scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=50,
    num_training_steps=max_train_steps,
)

unet, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
    unet, optimizer, train_dataloader, lr_scheduler
)

print("Total training steps:", max_train_steps)

In [ ]:
from tqdm.auto import tqdm

loss_history = []
global_step = 0
progress_bar = tqdm(range(max_train_steps), desc="Training")

for epoch in range(NUM_EPOCHS):
    unet.train()
    for step, batch in enumerate(train_dataloader):
        with accelerator.accumulate(unet):
            # 1. Encode images to latents (frozen VAE)
            pixel_values = batch["pixel_values"].to(device, dtype=weight_dtype)
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

            # 2. Sample noise and a random timestep per image
            noise = torch.randn_like(latents)
            bsz = latents.shape[0]
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device
            ).long()

            # 3. Add noise to latents according to the schedule (forward diffusion process)
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # 4. Encode captions (frozen CLIP text encoder)
            input_ids = batch["input_ids"].to(device)
            encoder_hidden_states = text_encoder(input_ids)[0]

            # 5. Predict the noise with the UNet (LoRA adapters are the only trainable part here)
            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

            # 6. Loss: how wrong was the noise prediction?
            loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")

            accelerator.backward(loss)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        if accelerator.sync_gradients:
            global_step += 1
            progress_bar.update(1)
            loss_history.append(loss.detach().item())
            if global_step % 20 == 0:
                progress_bar.set_postfix(loss=loss.detach().item(), epoch=epoch)

print("Training complete.")

## 6. Monitor the run — this is where most real training jobs are diagnosed

In production you'd log this to Weights & Biases or MLflow per the earlier note, but the diagnostic logic is the same either way:

- **Smooth downward trend** → healthy
- **Loss stuck flat near the start value** → learning rate too low, or LoRA rank too small for the task
- **Loss spikes/NaNs** → learning rate too high, or numerical instability from fp16 (the exact failure mode mentioned in the Modal Flux LoRA writeup we referenced — "high GPU utilization but produced only black images")
- **Loss goes to near-zero very fast** → likely overfitting/memorizing on a small dataset; check generated samples for loss of diversity, not just the loss number

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10, 4))
plt.plot(loss_history, alpha=0.3, label="raw loss")
# rolling average — raw diffusion loss is noisy step-to-step, the trend is what matters
window = 20
if len(loss_history) > window:
    rolling = np.convolve(loss_history, np.ones(window)/window, mode="valid")
    plt.plot(range(window-1, len(loss_history)), rolling, label=f"{window}-step rolling avg", linewidth=2)
plt.xlabel("Training step")
plt.ylabel("MSE noise-prediction loss")
plt.title("Training loss")
plt.legend()
plt.show()

## 7. Save the LoRA adapter

Notice the file size when this is done — this is the entire point of LoRA. You're not saving a multi-GB checkpoint, you're saving a small adapter that gets applied on top of the public base model.

In [ ]:
OUTPUT_DIR = "/content/pokemon-sd15-lora"
unwrapped_unet = accelerator.unwrap_model(unet)
unwrapped_unet.save_pretrained(OUTPUT_DIR)

import os
total_size = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f))
    for f in os.listdir(OUTPUT_DIR)
    if os.path.isfile(os.path.join(OUTPUT_DIR, f))
)
print(f"Saved LoRA adapter to {OUTPUT_DIR}")
print(f"Adapter size: {total_size / 1e6:.1f} MB  (compare: full SD1.5 UNet checkpoint is ~3.4 GB)")

## 8. Inference — load the base model + your adapter, generate

This is the deployment-shaped step: load the public base model once, then hot-load whichever LoRA adapter you want on top. In production this is exactly how you'd serve multiple fine-tuned "styles" or "domains" off one base model deployment (e.g. via vLLM's LoRA hot-swap for text models, or multiple diffusers LoRA adapters for image models) without duplicating the multi-GB base weights per customer/use case.

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to("cuda")

# Load your trained LoRA adapter on top of the base pipeline
pipe.load_lora_weights(OUTPUT_DIR)

prompt = "a cute fire-type creature with wings, pixel art style"
image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
image

In [ ]:
# Compare against the base model with the SAME prompt and seed — this is your actual evaluation
import matplotlib.pyplot as plt

generator = torch.Generator("cuda").manual_seed(0)

pipe.disable_lora()
base_image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=generator).images[0]

generator = torch.Generator("cuda").manual_seed(0)
pipe.enable_lora()
lora_image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=generator).images[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(base_image); axes[0].set_title("Base SD 1.5"); axes[0].axis("off")
axes[1].imshow(lora_image); axes[1].set_title("Your fine-tuned LoRA"); axes[1].axis("off")
plt.suptitle(prompt)
plt.show()

## 9. What to actually do next (in order)

1. **Run this as-is first.** Get a working baseline before changing anything.
2. **Vary `r` (rank)** — try 4, 8, 32. Watch how trainable-param-count and output quality/diversity change. This is the single most important hyperparameter to build intuition for.
3. **Vary learning rate** — try 5e-5 and 5e-4. Watch the loss curve shape change (too high → spiky/divergent; too low → flat).
4. **Swap in your own dataset.** This is the real test of whether you understand the pipeline. Build a folder of (image, caption) pairs in the same `imagefolder` + `metadata.csv` format HuggingFace `datasets` expects, push it to the Hub or load locally, and point `load_dataset` at it instead. Concretely: pick 100–300 images of *anything coherent* — a specific art style, a product category, your own photography — and write one-sentence captions for each.
5. **Read `diffusers/examples/text_to_image/train_text_to_image_lora.py` on GitHub.** That's the actual production script this notebook is a transparent, annotated version of — once this notebook makes sense, that script will be readable instead of intimidating.
6. **Move to SDXL with LoRA** once you have Colab Pro / an A100, using the exact same conceptual pipeline — the only real changes are the model name, two text encoders instead of one, and a higher native resolution (1024 instead of 512).
7. **Then do the LLM track** (QLoRA + SFT on a text dataset) — same underlying pattern: frozen base model, small trainable adapter, domain-specific data, monitor loss, evaluate before/after.